In [ ]:
%config InteractiveShell.cache_size = 0
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
from lucent.modelzoo import inceptionv1
import matplotlib.pyplot as plt

device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


plt.style.use("dark_background")

In [ ]:
from pathlib import Path
from torch.nn import functional as F

im_base = Path("flat-images")
w = model.get_submodule("mixed4e_1x1_pre_relu_conv").weight[55].detach().reshape(-1)

In [ ]:
from olt.tfms import transform

In [ ]:
ps = list(im_base.glob("*.jpeg"))
ps[:5]

In [ ]:
import torch
from olt.act import InputOutputModelSnapshot
from tqdm import tqdm
from PIL import Image

sims = []

bs = 16
model = model.to("mps")

above_0_3_acts = []
lbl_5_acts = []

with torch.no_grad():
    for i in tqdm(list(range(0, len(ps), bs))):
        batch = torch.stack([transform(Image.open(p)) for p in ps[i:i+bs]]).to("mps")
        
        # timg = transform(Image.open(p))[None]
        # print(i)
        act = InputOutputModelSnapshot.get_activations(batch, model, ["mixed4e_1x1_pre_relu_conv"])
        
        input_act = act["mixed4e_1x1_pre_relu_conv"]["input"] # [b,c,h,w]
        input_act = input_act.permute(1,0,2,3) # [c,b,h,w]
        input_act = input_act.reshape(input_act.shape[0], -1) #[c,positions]
        input_act = input_act.permute(1,0) # [positions,c]
        batch_sim = F.cosine_similarity(input_act, w) # [positions]
        batch_labels = km.predict(batch_sim.numpy().reshape(-1,1))
        for j, label in enumerate(batch_labels):
            # print(label)
            if label == 5:
                lbl_5_acts.append(input_act[j])
                
        # indices = torch.where(batch_sim > 0.3)
        # if len(indices) > 0:
        #     above_0_3_acts.append(input_act[indices])
        #     break
        sims.append(batch_sim)
        

In [ ]:
from olt.show import show_single_channel_red_green_black as S

In [ ]:
S([(w * lbl_5_acts[0]).reshape(22,24), (w * lbl_5_acts[1]).reshape(22,24)])
plt.show()

In [ ]:
S([(w * above_0_3_acts[0][0]).reshape(22,24), (w * above_0_3_acts[0][1]).reshape(22,24)])
plt.show()

In [ ]:
above_0_3_acts[0].shape

In [ ]:
csims = torch.load("cosine_sims.pt", weights_only=False)
csims = torch.cat(csims)
csims = torch.sort(csims).values

In [ ]:
torch.save(sims, "cosine_sims.pt")

In [ ]:
sims = torch.cat(sims)

In [ ]:
sorted_sims = torch.sort(sims).values

In [ ]:
sorted_sims[sorted_sims > 0.3]

In [ ]:
# very few, we can basically get the ones which are higher and take them,
# we do this later though. for now. peace

sorted_sims[sorted_sims > 0.3].shape

In [ ]:
plt.plot(csims)
plt.show()

In [ ]:
# lets make a distance matrix using cosine similarity
# and do a gaussian mixture model
# actually, we have it lol. i just need to find the correct components in gmm.
# kmeans might also be fine actually

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
csims = csims.numpy()

In [ ]:
csims

In [ ]:
ins = []
for c in tqdm(range(2, 100, 5)):
    # print(c)
    km = KMeans(c)
    km.fit(csims.reshape(-1,1))
    ins.append(km.inertia_)

In [ ]:
plt.plot(ins)

In [ ]:
ins = []
for c in tqdm(range(2, 20, 1)):
    # print(c)
    km = KMeans(c)
    km.fit(csims.reshape(-1,1))
    ins.append(km.inertia_)

In [ ]:
plt.plot(ins)

In [ ]:
km = KMeans(10).fit(csims.reshape(-1,1))

In [ ]:
X = csims
kmeans = km
labels = km.labels_

In [ ]:
idxs = np.argwhere((csims > 0.1) & (csims < 0.2))

In [ ]:
np.argmax(csims)

In [ ]:
labels[6271999]

In [ ]:
# see label 8 then
labels[idxs]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

X = X.flatten() if X.ndim > 1 else X  # ensure 1D

plt.figure(figsize=(8, 4))
plt.scatter(X, np.zeros_like(X), c=labels, cmap='viridis', s=50)
plt.yticks([])
plt.xlabel('Value')
plt.title('K-Means Clusters (1D)')


plt.show()